# 💻 Content-Based Recommendations (Concatenated Cosine - Single Vector)

Notebook này minh họa việc xây dựng hệ thống gợi ý bằng cách sử dụng **Cosine Tổng hợp (Concatenated Cosine Similarity)**. Tất cả các đặc trưng (Tác giả, Thể loại, NXB) được nối thành một vector duy nhất để tính độ tương đồng:

$$ S'(u,i) = \cos(\mathbf{U}_{\text{concat}}, \mathbf{I}_{\text{concat}}) = \frac{\mathbf{U}_{\text{concat}} \cdot \mathbf{I}_{\text{concat}}}{\| \mathbf{U}_{\text{concat}} \| \times \| \mathbf{I}_{\text{concat}} \| } $$


## 1) Cấu hình & Kết nối MySQL

In [1]:
# === CONFIG ===
MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_USER = "root"
MYSQL_PASSWORD = "123456"

DB_USER  = "bookweb_user"
DB_BOOK  = "bookweb_book"
DB_ORDER = "bookweb_order"

# In nhanh
PRINT_MAX_ITEMS = 10
PRINT_MAX_USERS = 5
TOP_K = 8

# === Thư viện ===
import numpy as np, pandas as pd
from sqlalchemy import create_engine, text
from collections import defaultdict

def get_engine():
    dsn = f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}"
    return create_engine(dsn, pool_pre_ping=True, pool_recycle=1800)

def fetch_all(sql: str, params: dict | None = None):
    with engine.connect() as conn:
        res = conn.execute(text(sql), params or {})
        cols = res.keys()
        return [dict(zip(cols, row)) for row in res.fetchall()]

engine = get_engine()
print("Kết nối CSDL đã sẵn sàng.")


Kết nối CSDL đã sẵn sàng.


## 2) Nạp dữ liệu

In [2]:
authors = fetch_all(f"SELECT id, name FROM {DB_BOOK}.authors")
categories = fetch_all(f"SELECT id, name FROM {DB_BOOK}.categories")
publishers = fetch_all(f"SELECT id, name FROM {DB_BOOK}.publishers")

author2idx = {r['id']: i for i, r in enumerate(authors)}
cate2idx   = {r['id']: i for i, r in enumerate(categories)}
pub2idx    = {r['id']: i for i, r in enumerate(publishers)}

idx2author = {i: r['name'] for i, r in enumerate(authors)}
idx2cate   = {i: r['name'] for i, r in enumerate(categories)}
idx2pub    = {i: r['name'] for i, r in enumerate(publishers)}

ab = fetch_all(f"SELECT book_id, author_id    FROM {DB_BOOK}.author_book")
bc = fetch_all(f"SELECT book_id, category_id FROM {DB_BOOK}.book_category")
bp = fetch_all(f"SELECT id AS book_id, publisher_id, title FROM {DB_BOOK}.books")

book_authors = defaultdict(list)
for r in ab:
    if r['author_id'] in author2idx:
        book_authors[r['book_id']].append(author2idx[r['author_id']])

book_cates = defaultdict(list)
for r in bc:
    if r['category_id'] in cate2idx:
        book_cates[r['book_id']].append(cate2idx[r['category_id']])

book_pubs = {}
book_meta = {}
book_ids = []
for r in bp:
    bid = r['book_id']
    book_ids.append(bid)
    if r['publisher_id'] in pub2idx:
        book_pubs[bid] = pub2idx[r['publisher_id']]
    book_meta[bid] = {'title': r['title']}

na, nc, npub = len(author2idx), len(cate2idx), len(pub2idx)
print("Kích thước danh mục:", "Authors=", na, "Categories=", nc, "Publishers=", npub, "| Books=", len(book_ids))


Kích thước danh mục: Authors= 18 Categories= 15 Publishers= 10 | Books= 30


## 3) Item profile
Ma trận item nhị phân cho từng khối (A_items, C_items, P_items)


In [3]:
def build_item_matrices():
    A = np.zeros((len(book_ids), na), dtype=float)
    C = np.zeros((len(book_ids), nc), dtype=float)
    P = np.zeros((len(book_ids), npub), dtype=float)
    for row_idx, bid in enumerate(book_ids):
        for ai in book_authors.get(bid, []): A[row_idx, ai] = 1.0
        for ci in book_cates.get(bid, []):     C[row_idx, ci] = 1.0
        pj = book_pubs.get(bid, None)
        if pj is not None: P[row_idx, pj] = 1.0
    return A, C, P

A_items, C_items, P_items = build_item_matrices()

A_cols = [f"A:{idx2author[i]}" for i in range(na)]
C_cols = [f"C:{idx2cate[i]}"    for i in range(nc)]
P_cols = [f"P:{idx2pub[i]}"     for i in range(npub)]

def df_items_block(M, cols):
    return pd.DataFrame(
        M[:PRINT_MAX_ITEMS, :],
        index=[f"{book_ids[i]}:{book_meta[book_ids[i]]['title']}" for i in range(min(PRINT_MAX_ITEMS, len(book_ids)))],
        columns=cols
    )

print("A_items:", A_items.shape, "| C_items:", C_items.shape, "| P_items:", P_items.shape)
df_items_block(A_items, A_cols).loc[:, (A_items[:PRINT_MAX_ITEMS]!=0).any(axis=0)]


A_items: (30, 18) | C_items: (30, 15) | P_items: (30, 10)


,A:Tô Hoài,A:Paulo CoeHo,A:Dale Carnegie,A:Rosie Nguyễn,A:Patrick Modiano,A:Hyun-wook park,A:Xuân Quỳnh
1:Dế Mèn phưu lưu kí,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2:Hoàng Tử Bé,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3:Đắc nhân tâm,0.0,0.0,1.0,0.0,0.0,0.0,0.0
10:Công chúa ngủ trong rừng,1.0,0.0,0.0,0.0,0.0,0.0,0.0
14:Tuổi trẻ đáng giá bao nhiêu?,0.0,0.0,0.0,1.0,0.0,0.0,0.0
15:Tuần trăng mật,0.0,0.0,0.0,0.0,1.0,0.0,0.0
16:Giã từ thơ ngây,0.0,0.0,0.0,0.0,0.0,1.0,0.0
17:Hồi ức là cuộn băng tua ngược,0.0,0.0,0.0,0.0,0.0,1.0,0.0
18:TRONG ĐÁY MẮT TRỜI XANH LÀ VĨNH VIỄN,0.0,0.0,0.0,0.0,0.0,0.0,1.0
19:KHÔNG BAO GIỜ LÀ CUỐI,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [4]:
df_items_block(C_items, C_cols).loc[:, (C_items[:PRINT_MAX_ITEMS]!=0).any(axis=0)]


,C:Tiểu thuyết,C:Hài hước,C:Kỹ năng,C:Thiếu nhi,C:Lãng mạn,C:Thơ - Kịch
1:Dế Mèn phưu lưu kí,1.0,0.0,0.0,1.0,0.0,0.0
2:Hoàng Tử Bé,1.0,1.0,0.0,0.0,0.0,0.0
3:Đắc nhân tâm,0.0,0.0,1.0,0.0,0.0,0.0
10:Công chúa ngủ trong rừng,0.0,0.0,0.0,1.0,0.0,0.0
14:Tuổi trẻ đáng giá bao nhiêu?,0.0,0.0,1.0,0.0,0.0,0.0
15:Tuần trăng mật,1.0,1.0,0.0,0.0,0.0,0.0
16:Giã từ thơ ngây,1.0,0.0,0.0,0.0,1.0,0.0
17:Hồi ức là cuộn băng tua ngược,1.0,1.0,0.0,0.0,1.0,0.0
18:TRONG ĐÁY MẮT TRỜI XANH LÀ VĨNH VIỄN,0.0,0.0,0.0,0.0,0.0,1.0
19:KHÔNG BAO GIỜ LÀ CUỐI,0.0,0.0,0.0,0.0,0.0,1.0


In [5]:
df_items_block(P_items, P_cols).loc[:, (P_items[:PRINT_MAX_ITEMS]!=0).any(axis=0)]


,P:Kim Đồng,P:Nhã Nam,P:NXB Trẻ,P:Hội nhà văn
1:Dế Mèn phưu lưu kí,1.0,0.0,0.0,0.0
2:Hoàng Tử Bé,1.0,0.0,0.0,0.0
3:Đắc nhân tâm,0.0,0.0,1.0,0.0
10:Công chúa ngủ trong rừng,1.0,0.0,0.0,0.0
14:Tuổi trẻ đáng giá bao nhiêu?,0.0,0.0,1.0,0.0
15:Tuần trăng mật,0.0,0.0,0.0,1.0
16:Giã từ thơ ngây,0.0,1.0,0.0,0.0
17:Hồi ức là cuộn băng tua ngược,0.0,0.0,1.0,0.0
18:TRONG ĐÁY MẮT TRỜI XANH LÀ VĨNH VIỄN,0.0,0.0,0.0,1.0
19:KHÔNG BAO GIỜ LÀ CUỐI,0.0,0.0,0.0,1.0


## 4) Ma trận **W** (user–item) — review & favorite

Ma trận tương tác W (Trọng số Rating) được xây dựng từ dữ liệu reviews và favorites.

In [6]:
# Tập user từ reviews ∪ favorites
uids_reviews = fetch_all(f"SELECT DISTINCT buyer_id AS uid FROM {DB_ORDER}.reviews")
uids_favs    = fetch_all(f"SELECT DISTINCT buyer_id AS uid FROM {DB_ORDER}.favorites")
user_ids = sorted(list({r['uid'] for r in (uids_reviews + uids_favs)}))

uid2row = {uid: i for i, uid in enumerate(user_ids)}
bid2col = {bid: j for j, bid in enumerate(book_ids)}

W = np.zeros((len(user_ids), len(book_ids)), dtype=float)

# Review mới nhất cho mỗi (user, book)
latest_reviews = fetch_all(f"""
    SELECT buyer_id AS uid, book_id, stars AS rating
    FROM (
        SELECT buyer_id, book_id, stars,
               ROW_NUMBER() OVER (PARTITION BY buyer_id, book_id ORDER BY created_at DESC, id DESC) rn
        FROM {DB_ORDER}.reviews
    ) t
    WHERE rn = 1
""")
for r in latest_reviews:
    uid, bid, rating = r['uid'], r['book_id'], float(r['rating'])
    if uid in uid2row and bid in bid2col:
        W[uid2row[uid], bid2col[bid]] = max(W[uid2row[uid], bid2col[bid]], rating)

# Favorites = 5 điểm
favs = fetch_all(f"SELECT buyer_id AS uid, book_id FROM {DB_ORDER}.favorites")
for r in favs:
    uid, bid = r['uid'], r['book_id']
    if uid in uid2row and bid in bid2col:
        W[uid2row[uid], bid2col[bid]] = max(W[uid2row[uid], bid2col[bid]], 5.0)

print("W shape:", W.shape, "| #users:", len(user_ids), "| #items:", len(book_ids))

# In vài hàng đầu (ẩn cột toàn 0)
dfW = pd.DataFrame(
    W[:PRINT_MAX_USERS, :PRINT_MAX_ITEMS],
    index=[f"u{i}:{user_ids[i]}" for i in range(min(PRINT_MAX_USERS, len(user_ids)))],
    columns=[f"{book_ids[j]}:{book_meta[book_ids[j]]['title']}" for j in range(min(PRINT_MAX_ITEMS, len(book_ids)))]
)
dfW.loc[:, (dfW.abs() > 0).any(axis=0)]


W shape: (11, 30) | #users: 11 | #items: 30


,1:Dế Mèn phưu lưu kí,14:Tuổi trẻ đáng giá bao nhiêu?
u0:11175e62-c856-46fa-b705-fb8ee2fbf441,0.0,0.0
u1:33375e62-c856-46fa-b705-fb8ee2fbf333,0.0,0.0
u2:44475e62-c856-46fa-b705-fb8ee2fbf444,0.0,0.0
u3:55575e62-c856-46fa-b705-fb8ee2fbf555,0.0,0.0
u4:66675e62-c856-46fa-b705-fb8ee2fbf666,4.0,3.0


## 5) Vector user Raw (Tổng điểm thô)

Tính **Tổng điểm** sở thích cho từng thuộc tính: $\mathbf{U}_{\text{raw}} = \mathbf{W} \times \mathbf{U}_{\text{items}}$.

Lưu ý: Các vector $\mathbf{A}_{\text{user}}, \mathbf{C}_{\text{user}}, \mathbf{P}_{\text{user}}$ ở đây là các vector **thô (raw sum)** và sẽ được nối lại trong **Mục 5.5**.


In [7]:
# Tính Vector Sở thích Thô (Raw User Profiles - Tổng điểm)
A_user = W @ A_items
C_user = W @ C_items
P_user = W @ P_items

print("A_user:", A_user.shape, "| C_user:", C_user.shape, "| P_user:", P_user.shape)

def df_user_block(M, cols, block_name):
    df = pd.DataFrame(
        M[:PRINT_MAX_USERS, :],
        index=[f"u{i}:{user_ids[i]}" for i in range(min(PRINT_MAX_USERS, len(user_ids)))],
        columns=cols
    )
    print(f"\n--- VÍ DỤ: {block_name} (Tổng điểm Thô) của 5 User đầu tiên ---")
    return df.loc[:, (df.abs() > 0).any(axis=0)]

df_user_block(A_user, A_cols, "Vector A_user - Tác giả")


A_user: (11, 18) | C_user: (11, 15) | P_user: (11, 10)

--- VÍ DỤ: Vector A_user - Tác giả (Tổng điểm Thô) của 5 User đầu tiên ---


,A:Tô Hoài,A:Rosie Nguyễn,A:J. R. R. Tolkien,A:Laura Cowan,A:Stephen Hawking,A: Phương Hoài Nga,A: Alexandre Dumas,A:Trần Lỗi,A:Benjamin Graham,A:Raymond Chandler
u0:11175e62-c856-46fa-b705-fb8ee2fbf441,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0
u1:33375e62-c856-46fa-b705-fb8ee2fbf333,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0
u2:44475e62-c856-46fa-b705-fb8ee2fbf444,0.0,0.0,0.0,5.0,0.0,5.0,0.0,0.0,4.0,0.0
u3:55575e62-c856-46fa-b705-fb8ee2fbf555,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0
u4:66675e62-c856-46fa-b705-fb8ee2fbf666,4.0,3.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0


In [8]:
df_user_block(C_user, C_cols, "Vector C_user - Thể loại")



--- VÍ DỤ: Vector C_user - Thể loại (Tổng điểm Thô) của 5 User đầu tiên ---


,C:Tiểu thuyết,C:Hài hước,C:Kỹ năng,C:Thiếu nhi,C:Giáo trình,C:Khoa học,C:Văn học,C:Phiêu lưu,C:Kiến thức,C:Truyện tranh,C:Tài chính - Kinh doanh,C:Trinh thám
u0:11175e62-c856-46fa-b705-fb8ee2fbf441,0.0,5.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,5.0,0.0,0.0
u1:33375e62-c856-46fa-b705-fb8ee2fbf333,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,5.0,0.0,0.0,0.0
u2:44475e62-c856-46fa-b705-fb8ee2fbf444,0.0,0.0,9.0,5.0,5.0,5.0,0.0,0.0,14.0,0.0,4.0,0.0
u3:55575e62-c856-46fa-b705-fb8ee2fbf555,10.0,0.0,0.0,0.0,0.0,0.0,5.0,5.0,0.0,0.0,0.0,5.0
u4:66675e62-c856-46fa-b705-fb8ee2fbf666,4.0,4.0,3.0,4.0,0.0,0.0,0.0,4.0,0.0,4.0,0.0,0.0


In [9]:
df_user_block(P_user, P_cols, "Vector P_user - NXB")



--- VÍ DỤ: Vector P_user - NXB (Tổng điểm Thô) của 5 User đầu tiên ---


,P:Kim Đồng,P:NXB Trẻ,P:Hội nhà văn,P:Lao Động,P:Văn học,P:Dân Trí,P:Thế Giới,P:Phụ Nữ
u0:11175e62-c856-46fa-b705-fb8ee2fbf441,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0
u1:33375e62-c856-46fa-b705-fb8ee2fbf333,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0
u2:44475e62-c856-46fa-b705-fb8ee2fbf444,0.0,0.0,0.0,4.0,0.0,5.0,0.0,5.0
u3:55575e62-c856-46fa-b705-fb8ee2fbf555,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0
u4:66675e62-c856-46fa-b705-fb8ee2fbf666,4.0,3.0,4.0,0.0,0.0,0.0,0.0,0.0


## 5.5) Nối Vector (Concatenation)

Nối các vector sở thích/đặc trưng theo khối thành một vector dài duy nhất ($\mathbf{U}_{\text{concat}}, \mathbf{I}_{\text{concat}}$). Phép tính Cosine sau đó sẽ được thực hiện trên không gian đặc trưng tổng hợp này.


In [10]:
# Nối các User Profile (theo khối) lại với nhau
U_CONCAT = np.hstack([A_user, C_user, P_user])

# Nối các Item Feature (theo khối) lại với nhau
I_CONCAT = np.hstack([A_items, C_items, P_items])

print(f"U_CONCAT shape: {U_CONCAT.shape} | I_CONCAT shape: {I_CONCAT.shape}")

# Hiển thị một phần ma trận nối
ALL_COLS = A_cols + C_cols + P_cols
df_concat = pd.DataFrame(
    U_CONCAT[:PRINT_MAX_USERS, :],
    index=[f"u{i}:{user_ids[i]}" for i in range(min(PRINT_MAX_USERS, len(user_ids)))],
    columns=ALL_COLS
)
df_concat.loc[:, (df_concat.abs() > 0).any(axis=0)]


U_CONCAT shape: (11, 43) | I_CONCAT shape: (30, 43)


,A:Tô Hoài,A:Rosie Nguyễn,A:J. R. R. Tolkien,A:Laura Cowan,A:Stephen Hawking,A: Phương Hoài Nga,A: Alexandre Dumas,A:Trần Lỗi,A:Benjamin Graham,A:Raymond Chandler,...,C:Tài chính - Kinh doanh,C:Trinh thám,P:Kim Đồng,P:NXB Trẻ,P:Hội nhà văn,P:Lao Động,P:Văn học,P:Dân Trí,P:Thế Giới,P:Phụ Nữ
u0:11175e62-c856-46fa-b705-fb8ee2fbf441,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,...,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0
u1:33375e62-c856-46fa-b705-fb8ee2fbf333,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0
u2:44475e62-c856-46fa-b705-fb8ee2fbf444,0.0,0.0,0.0,5.0,0.0,5.0,0.0,0.0,4.0,0.0,...,4.0,0.0,0.0,0.0,0.0,4.0,0.0,5.0,0.0,5.0
u3:55575e62-c856-46fa-b705-fb8ee2fbf555,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,...,0.0,5.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0
u4:66675e62-c856-46fa-b705-fb8ee2fbf666,4.0,3.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,...,0.0,0.0,4.0,3.0,4.0,0.0,0.0,0.0,0.0,0.0


## 6) Cosine Tổng hợp (Tính điểm S(u,i))

Thực hiện tính Cosine Similarity một lần duy nhất trên các vector đã nối $\mathbf{U}_{\text{concat}}$ và $\mathbf{I}_{\text{concat}}$.


In [11]:
def safe_cos(u_vec: np.ndarray, i_vec: np.ndarray) -> float:
    nu = np.linalg.norm(u_vec); ni = np.linalg.norm(i_vec)
    if nu == 0.0 or ni == 0.0:
        return 0.0
    # Hàm này tính Cosine Similarity trên vector đã nối
    return float(u_vec.dot(i_vec) / (nu * ni))

def score_user_item_concat(u_idx: int, i_idx: int) -> float:
    # Lấy vector đã nối của user và item
    u_vec = U_CONCAT[u_idx, :]
    i_vec = I_CONCAT[i_idx, :]
    
    # Chỉ tính một điểm Cosine duy nhất
    return safe_cos(u_vec, i_vec)

def build_scores_matrix():
    m, n = W.shape
    S = np.zeros((m, n), dtype=float)
    for u in range(m):
        for i in range(n):
            S[u, i] = score_user_item_concat(u, i)
    return S

S = build_scores_matrix()
print("S shape:", S.shape)

dfS = pd.DataFrame(
    S[:PRINT_MAX_USERS, :PRINT_MAX_ITEMS],
    index=[f"u{i}:{user_ids[i]}" for i in range(min(PRINT_MAX_USERS, len(user_ids)))],
    columns=[f"{book_ids[j]}:{book_meta[book_ids[j]]['title']}" for j in range(min(PRINT_MAX_ITEMS, len(book_ids)))]
)
dfS.loc[:, (dfS.abs() > 0).any(axis=0)]


S shape: (11, 30)


,1:Dế Mèn phưu lưu kí,2:Hoàng Tử Bé,3:Đắc nhân tâm,10:Công chúa ngủ trong rừng,14:Tuổi trẻ đáng giá bao nhiêu?,15:Tuần trăng mật,16:Giã từ thơ ngây,17:Hồi ức là cuộn băng tua ngược,18:TRONG ĐÁY MẮT TRỜI XANH LÀ VĨNH VIỄN,19:KHÔNG BAO GIỜ LÀ CUỐI
u0:11175e62-c856-46fa-b705-fb8ee2fbf441,0.000000,0.223607,0.258199,0.000000,0.258199,0.223607,0.000000,0.400000,0.000000,0.000000
u1:33375e62-c856-46fa-b705-fb8ee2fbf333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
u2:44475e62-c856-46fa-b705-fb8ee2fbf444,0.111803,0.000000,0.232379,0.129099,0.232379,0.000000,0.000000,0.000000,0.000000,0.000000
u3:55575e62-c856-46fa-b705-fb8ee2fbf555,0.277350,0.277350,0.000000,0.000000,0.000000,0.277350,0.277350,0.248069,0.000000,0.000000
u4:66675e62-c856-46fa-b705-fb8ee2fbf666,0.611775,0.458831,0.264906,0.529813,0.397360,0.458831,0.152944,0.376192,0.176604,0.176604


## 7) Recommend Top-K (loại item đã tương tác)

In [12]:
def recommend_for_user(u_idx: int, top_k=TOP_K, exclude_interacted=True):
    m, n = W.shape
    inter = set(np.where(W[u_idx, :] > 0)[0]) if exclude_interacted else set()
    rows = []
    for j in range(n):
        if j in inter:
            continue
        rows.append((j, float(S[u_idx, j])))
    rows.sort(key=lambda x: x[1], reverse=True)
    rows = rows[:top_k]
    out = []
    for j, sc in rows:
        bid = book_ids[j]
        out.append({"book_id": bid, "title": book_meta[bid]["title"], "score": sc})
    return out

if len(user_ids) > 0:
    u0 = 2
    print(f"User u{u0}: {user_ids[u0]}")
    recs = recommend_for_user(u0, top_k=TOP_K, exclude_interacted=True)
    recs_df = pd.DataFrame(recs)
    recs_df


User u2: 44475e62-c856-46fa-b705-fb8ee2fbf444


## 8) Demo chi tiết (Concatenated Cosine)

Giải thích cách tính ra **ĐIỂM COSINE DUY NHẤT** dựa trên vector $\mathbf{U}_{\text{concat}}$ và $\mathbf{I}_{\text{concat}}$.

In [13]:
def list_interactions(u_idx: int):
    js = np.where(W[u_idx, :] > 0)[0]
    rows = []
    for j in js:
        bid = book_ids[j]
        rows.append({"book_id": bid, "title": book_meta[bid]["title"], "rating": float(W[u_idx, j])})
    rows.sort(key=lambda x: -x["rating"])
    return rows

def top_features_block(vec, names, k=10):
    # Lấy các feature khác 0 trong vector U_CONCAT
    nz = [(names[i], float(vec[i])) for i in np.where(vec != 0)[0]]
    nz.sort(key=lambda x: abs(x[1]), reverse=True)
    return nz[:k]

def get_vectors_nonzero(u_vec, i_vec, names):
    # Lấy các chỉ mục (index) nơi ít nhất một vector khác 0
    nz_indices = np.where((u_vec != 0) | (i_vec != 0))[0]
    
    output = []
    for idx in nz_indices:
        output.append({
            'name': names[idx],
            'u_val': u_vec[idx],
            'i_val': i_vec[idx]
        })
    return output

def explain_user(u_idx: int, k_feat=10, sample_item_idx=None):
    print(f"=== USER u{u_idx}: {user_ids[u_idx]} ===")
    rows = list_interactions(u_idx)
    print("\n1. Các sách đã tương tác:")
    for r in rows:
        print(f"  - {r['rating']:>4.1f} × [{r['book_id']}] {r['title']}")

    u_concat = U_CONCAT[u_idx, :]
    
    print("\n2. Top features in U_CONCAT (Tổng điểm thô - Toàn bộ đặc trưng):")
    for name, val in top_features_block(u_concat, ALL_COLS, k=k_feat):
        print(f"  - {name:40s} {val: .3f}")

    if sample_item_idx is not None:
        i_concat = I_CONCAT[sample_item_idx, :]
        
        dot_score = u_concat.dot(i_concat)
        norm_u = np.linalg.norm(u_concat)
        norm_i = np.linalg.norm(i_concat)
        final_score = safe_cos(u_concat, i_concat)
        
        bid = book_ids[sample_item_idx]
        
        print(f"\n3. SCORE CALCULATION for item [{bid}] {book_meta[bid]['title']}:")
        print("\n	--- CÔNG THỨC COSINE TỔNG HỢP --- ")
        
        # In các thành phần đóng góp vào Dot Product
        vec_nz = get_vectors_nonzero(u_concat, i_concat, ALL_COLS)
        print("\tCác thuộc tính đóng góp (u_val * i_val):")
        dot_calc = []
        for v in vec_nz:
            print(f"\t	- {v['name']:30s} | User: {v['u_val']:.3f} | Item: {v['i_val']:.0f}")
            if v['i_val'] != 0 and v['u_val'] != 0:
                dot_calc.append(f"({v['u_val']:.3f} * {v['i_val']:.0f})")
        
        print(f"\n	=> Dot product (u.i): {' + '.join(dot_calc)} = {dot_score:.4f}")
        print(f"\t- Norm user (||u||): {norm_u:.4f} ($\sqrt{{\sum u_i^2}}$)")
        print(f"\t- Norm item (||i||): {norm_i:.4f} ($\sqrt{{\sum i_i^2}}$)")
        
        print(f"\n	FINAL SCORE S(u,i) = {dot_score:.4f} / ({norm_u:.4f} \times {norm_i:.4f}) \approx \mathbf{{{final_score:.6f}}}")

if len(user_ids) > 0:
    u0 = 1
    # Lấy mục được gợi ý cao nhất từ bước 7
    recs = recommend_for_user(u0, top_k=TOP_K, exclude_interacted=True)
    sample_idx = None
    if len(recs) > 0:
        b2j = {book_ids[j]: j for j in range(len(book_ids))}
        sample_idx = b2j[recs[0]["book_id"]]
    explain_user(u0, k_feat=10, sample_item_idx=sample_idx)


=== USER u1: 33375e62-c856-46fa-b705-fb8ee2fbf333 ===

1. Các sách đã tương tác:
  -  5.0 × [27] MỞ KHÓA VŨ TRỤ

2. Top features in U_CONCAT (Tổng điểm thô - Toàn bộ đặc trưng):
  - A:Stephen Hawking                         5.000
  - C:Khoa học                                5.000
  - C:Kiến thức                               5.000
  - P:Thế Giới                                5.000

3. SCORE CALCULATION for item [32] MỌI ĐIỀU BẠN CẦN BIẾT VỀ VŨ TRỤ:

	--- CÔNG THỨC COSINE TỔNG HỢP --- 
	Các thuộc tính đóng góp (u_val * i_val):
		- A:Stephen Hawking              | User: 5.000 | Item: 0
		- A:Chris Cooper                 | User: 0.000 | Item: 1
		- C:Khoa học                     | User: 5.000 | Item: 1
		- C:Kiến thức                    | User: 5.000 | Item: 1
		- P:Thế Giới                     | User: 5.000 | Item: 1

	=> Dot product (u.i): (5.000 * 1) + (5.000 * 1) + (5.000 * 1) = 15.0000
	- Norm user (||u||): 10.0000 ($\sqrt{\sum u_i^2}$)
	- Norm item (||i||): 2.0000 ($\sqrt{\sum i_i